# 04_05: Document Understanding

**Objective:** Learn to extract structured information from documents — forms, receipts, invoices, and tables — using multimodal models that understand both text and layout.

**Prerequisites:** Familiarity with HuggingFace pipelines (Notebook 00_03), OCR concepts (Notebook 02_03).

## Prerequisites

### Hardware Requirements

| Model Option | Model Name | Size | Min RAM | Recommended Setup | Notes |
|-------------|------------|------|---------|-------------------|-------|
| **Small (CPU)** | impira/layoutlm-document-qa | ~440MB | 4GB RAM | Any CPU | Document QA |
| **Medium (CPU)** | naver-clova-ix/donut-base-finetuned-docvqa | ~800MB | 8GB RAM | Any CPU | End-to-end, no OCR needed |
| **Large (GPU)** | microsoft/layoutlmv3-base | ~500MB | 8GB RAM | RTX 4080+ | Layout-aware understanding |

## Expected Behaviors

### First Time Running
- **Model download**: ~440MB for LayoutLM (2-3 minutes)
- Donut is ~800MB if used

### What You'll See
- Models can answer questions about document content and layout
- LayoutLM uses OCR + layout information; Donut works end-to-end from images
- Structured fields (dates, amounts, names) are extracted with high accuracy

### Common Observations
- Clean, well-structured documents get better results than handwritten ones
- Layout matters: the same text in a header vs body may be interpreted differently
- Table extraction is still challenging for most models

## Overview

**Document Understanding** goes beyond OCR (which just extracts text) to understand the *meaning* and *structure* of documents. It combines:

1. **Visual features**: What does the document look like? (images, logos, formatting)
2. **Text content**: What words are on the page? (from OCR or embedded text)
3. **Layout information**: Where are the words positioned? (bounding boxes, spatial relationships)

### Model Comparison

| Model | Approach | OCR Required? | Strengths |
|-------|----------|---------------|----------|
| **LayoutLM** | Text + Layout + Image | Yes (provides OCR) | Fast, well-established |
| **LayoutLMv3** | Unified multimodal | Yes | State-of-the-art accuracy |
| **Donut** | End-to-end vision | No (reads directly) | No OCR dependency, simpler pipeline |

### Common Tasks

| Task | Description | Example |
|------|-------------|--------|
| **Document QA** | Answer questions about a document | "What is the total amount?" |
| **Key-Value Extraction** | Extract field-value pairs | Invoice number, date, amount |
| **Document Classification** | Categorize document type | Invoice, receipt, form, letter |
| **Table Extraction** | Parse tabular data | Extract rows and columns |

## Setup and Installation

In [ ]:
import sys
import time
import random
import warnings

import numpy as np
import pandas as pd
import torch
import transformers
from transformers import pipeline
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# Reproducibility
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Version metadata
print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'Transformers: {transformers.__version__}')
if torch.cuda.is_available():
    print(f'CUDA: {torch.version.cuda}')
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Model Selection

In [ ]:
# Option 1: LayoutLM Document QA (CPU-friendly, recommended)
MODEL_NAME = 'impira/layoutlm-document-qa'       # ~440MB

# Option 2: Donut (end-to-end, no OCR needed, larger)
# MODEL_NAME = 'naver-clova-ix/donut-base-finetuned-docvqa'  # ~800MB

## Creating Synthetic Documents

To demonstrate document understanding without requiring external PDFs or images, we'll create synthetic document images that mimic common document types. This also helps illustrate how layout affects understanding.

In [ ]:
def create_invoice_image(
    width: int = 600,
    height: int = 800,
) -> Image.Image:
    """Create a synthetic invoice document image.

    Args:
        width: Image width.
        height: Image height.

    Returns:
        PIL Image of a synthetic invoice.
    """
    image = Image.new('RGB', (width, height), 'white')
    draw = ImageDraw.Draw(image)
    
    # Try to use a basic font, fall back to default
    try:
        font_large = ImageFont.truetype('arial.ttf', 24)
        font_medium = ImageFont.truetype('arial.ttf', 16)
        font_small = ImageFont.truetype('arial.ttf', 12)
    except (OSError, IOError):
        font_large = ImageFont.load_default()
        font_medium = font_large
        font_small = font_large
    
    # Header
    draw.text((30, 20), 'INVOICE', fill='black', font=font_large)
    draw.line([(30, 55), (width - 30, 55)], fill='black', width=2)
    
    # Company info
    draw.text((30, 70), 'Acme Corporation', fill='black', font=font_medium)
    draw.text((30, 95), '123 Business Street', fill='gray', font=font_small)
    draw.text((30, 115), 'New York, NY 10001', fill='gray', font=font_small)
    
    # Invoice details
    draw.text((350, 70), 'Invoice #: INV-2024-0042', fill='black', font=font_small)
    draw.text((350, 90), 'Date: March 15, 2024', fill='black', font=font_small)
    draw.text((350, 110), 'Due Date: April 15, 2024', fill='black', font=font_small)
    
    # Bill To
    draw.text((30, 160), 'Bill To:', fill='black', font=font_medium)
    draw.text((30, 185), 'John Smith', fill='black', font=font_small)
    draw.text((30, 205), '456 Customer Avenue', fill='gray', font=font_small)
    draw.text((30, 225), 'Los Angeles, CA 90001', fill='gray', font=font_small)
    
    # Table header
    y_table = 280
    draw.rectangle([(30, y_table), (width - 30, y_table + 30)], fill='lightgray')
    draw.text((40, y_table + 5), 'Item', fill='black', font=font_small)
    draw.text((250, y_table + 5), 'Qty', fill='black', font=font_small)
    draw.text((330, y_table + 5), 'Price', fill='black', font=font_small)
    draw.text((450, y_table + 5), 'Total', fill='black', font=font_small)
    
    # Table rows
    items = [
        ('Web Development', '40 hrs', '$150.00', '$6,000.00'),
        ('UI/UX Design', '20 hrs', '$120.00', '$2,400.00'),
        ('Server Setup', '1', '$500.00', '$500.00'),
        ('Monthly Hosting', '3 mo', '$50.00', '$150.00'),
    ]
    
    for i, (item, qty, price, total) in enumerate(items):
        y = y_table + 35 + i * 30
        draw.text((40, y + 5), item, fill='black', font=font_small)
        draw.text((250, y + 5), qty, fill='black', font=font_small)
        draw.text((330, y + 5), price, fill='black', font=font_small)
        draw.text((450, y + 5), total, fill='black', font=font_small)
        draw.line([(30, y + 28), (width - 30, y + 28)], fill='lightgray')
    
    # Totals
    y_total = y_table + 35 + len(items) * 30 + 20
    draw.line([(350, y_total), (width - 30, y_total)], fill='black', width=1)
    draw.text((350, y_total + 10), 'Subtotal:', fill='black', font=font_small)
    draw.text((450, y_total + 10), '$9,050.00', fill='black', font=font_small)
    draw.text((350, y_total + 35), 'Tax (8%):', fill='black', font=font_small)
    draw.text((450, y_total + 35), '$724.00', fill='black', font=font_small)
    draw.line([(350, y_total + 60), (width - 30, y_total + 60)], fill='black', width=2)
    draw.text((350, y_total + 65), 'TOTAL:', fill='black', font=font_medium)
    draw.text((450, y_total + 65), '$9,774.00', fill='black', font=font_medium)
    
    # Payment info
    draw.text((30, height - 100), 'Payment Terms: Net 30', fill='gray', font=font_small)
    draw.text((30, height - 75), 'Bank: First National Bank', fill='gray', font=font_small)
    draw.text((30, height - 50), 'Account: XXXX-XXXX-1234', fill='gray', font=font_small)
    
    return image


# Create and display the invoice
invoice_image = create_invoice_image()
plt.figure(figsize=(8, 10))
plt.imshow(invoice_image)
plt.title('Synthetic Invoice Document', fontsize=13)
plt.axis('off')
plt.tight_layout()
plt.show()

## Method 1: Document QA Pipeline

The `document-question-answering` pipeline takes a document image and a question, then extracts the answer from the document. It uses OCR internally to read the text and layout information to understand structure.

In [ ]:
# Create Document QA pipeline
doc_qa_pipeline = pipeline(
    'document-question-answering',
    model=MODEL_NAME,
    device=device,
)

# Ask questions about the invoice
questions = [
    'What is the invoice number?',
    'What is the total amount?',
    'Who is the invoice billed to?',
    'What is the due date?',
    'What company issued this invoice?',
    'What is the tax amount?',
]

results: list[dict[str, str]] = []
for question in questions:
    try:
        answer = doc_qa_pipeline(
            image=invoice_image,
            question=question,
        )
        if isinstance(answer, list):
            answer = answer[0]
        results.append({
            'Question': question,
            'Answer': answer.get('answer', 'N/A'),
            'Score': f'{answer.get("score", 0):.4f}',
        })
    except Exception as error:
        results.append({
            'Question': question,
            'Answer': f'Error: {str(error)[:50]}',
            'Score': 'N/A',
        })

pd.DataFrame(results)

Document QA models are particularly powerful because they understand not just the text content but also the *spatial layout* of the document. A total amount at the bottom of a table is understood differently from the same number appearing in a description field. This layout awareness is what makes LayoutLM family models special compared to plain text QA.

## Testing with Different Document Types

Let's create a receipt to test the model on a different document format.

In [ ]:
def create_receipt_image(
    width: int = 350,
    height: int = 500,
) -> Image.Image:
    """Create a synthetic receipt document image.

    Args:
        width: Image width.
        height: Image height.

    Returns:
        PIL Image of a synthetic receipt.
    """
    image = Image.new('RGB', (width, height), 'white')
    draw = ImageDraw.Draw(image)
    
    try:
        font = ImageFont.truetype('arial.ttf', 14)
        font_small = ImageFont.truetype('arial.ttf', 11)
        font_bold = ImageFont.truetype('arialbd.ttf', 16)
    except (OSError, IOError):
        font = ImageFont.load_default()
        font_small = font
        font_bold = font
    
    y = 20
    # Header
    draw.text((width // 2 - 60, y), 'COFFEE HOUSE', fill='black', font=font_bold)
    y += 25
    draw.text((width // 2 - 70, y), '789 Main St, Seattle, WA', fill='gray', font=font_small)
    y += 20
    draw.text((width // 2 - 50, y), 'Tel: (206) 555-0123', fill='gray', font=font_small)
    y += 30
    draw.line([(20, y), (width - 20, y)], fill='black', width=1)
    y += 10
    
    draw.text((20, y), 'Date: 2024-03-15 10:30 AM', fill='black', font=font_small)
    y += 20
    draw.text((20, y), 'Order #: 1847', fill='black', font=font_small)
    y += 20
    draw.text((20, y), 'Cashier: Maria', fill='black', font=font_small)
    y += 25
    draw.line([(20, y), (width - 20, y)], fill='black', width=1)
    y += 10
    
    # Items
    items = [
        ('Cappuccino (L)', '$5.50'),
        ('Blueberry Muffin', '$3.75'),
        ('Avocado Toast', '$8.95'),
        ('Orange Juice', '$4.25'),
    ]
    for item, price in items:
        draw.text((20, y), item, fill='black', font=font)
        draw.text((width - 70, y), price, fill='black', font=font)
        y += 22
    
    y += 10
    draw.line([(20, y), (width - 20, y)], fill='black', width=1)
    y += 10
    
    draw.text((20, y), 'Subtotal:', fill='black', font=font)
    draw.text((width - 70, y), '$22.45', fill='black', font=font)
    y += 22
    draw.text((20, y), 'Tax (10%):', fill='black', font=font)
    draw.text((width - 70, y), '$2.25', fill='black', font=font)
    y += 22
    draw.line([(20, y), (width - 20, y)], fill='black', width=2)
    y += 5
    draw.text((20, y), 'TOTAL:', fill='black', font=font_bold)
    draw.text((width - 75, y), '$24.70', fill='black', font=font_bold)
    y += 30
    
    draw.text((20, y), 'Payment: Visa **** 4532', fill='black', font=font_small)
    y += 25
    draw.text((width // 2 - 50, y), 'Thank you!', fill='gray', font=font)
    
    return image


receipt_image = create_receipt_image()

fig, axes = plt.subplots(1, 2, figsize=(12, 8))
axes[0].imshow(invoice_image)
axes[0].set_title('Invoice', fontsize=13)
axes[0].axis('off')
axes[1].imshow(receipt_image)
axes[1].set_title('Receipt', fontsize=13)
axes[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Query the receipt
receipt_questions = [
    'What is the store name?',
    'What is the total?',
    'What items were ordered?',
    'What was the payment method?',
    'Who was the cashier?',
    'What is the order number?',
]

receipt_results: list[dict[str, str]] = []
for question in receipt_questions:
    try:
        answer = doc_qa_pipeline(image=receipt_image, question=question)
        if isinstance(answer, list):
            answer = answer[0]
        receipt_results.append({
            'Question': question,
            'Answer': answer.get('answer', 'N/A'),
            'Score': f'{answer.get("score", 0):.4f}',
        })
    except Exception as error:
        receipt_results.append({
            'Question': question,
            'Answer': f'Error: {str(error)[:50]}',
            'Score': 'N/A',
        })

print('=== Receipt QA ===')
pd.DataFrame(receipt_results)

## Practical Applications

Document understanding powers many real-world systems: invoice processing in accounting, form digitization in government, receipt scanning in expense tracking, and contract analysis in legal. Let's build a structured extraction pipeline.

In [ ]:
def extract_document_fields(
    image: Image.Image,
    fields: dict[str, str],
    doc_pipe: transformers.Pipeline,
) -> pd.DataFrame:
    """Extract specific fields from a document using QA.

    Args:
        image: Document image.
        fields: Dict mapping field names to extraction questions.
        doc_pipe: Document QA pipeline.

    Returns:
        DataFrame with extracted field values.
    """
    results: list[dict[str, str]] = []
    for field_name, question in fields.items():
        try:
            answer = doc_pipe(image=image, question=question)
            if isinstance(answer, list):
                answer = answer[0]
            confidence = answer.get('score', 0)
            status = 'Extracted' if confidence > 0.5 else 'Low confidence'
            results.append({
                'Field': field_name,
                'Value': answer.get('answer', 'N/A'),
                'Confidence': f'{confidence:.4f}',
                'Status': status,
            })
        except Exception:
            results.append({
                'Field': field_name,
                'Value': 'Extraction failed',
                'Confidence': '0.0000',
                'Status': 'Error',
            })
    return pd.DataFrame(results)


# Define invoice extraction schema
invoice_fields = {
    'Invoice Number': 'What is the invoice number?',
    'Vendor Name': 'What company issued this invoice?',
    'Customer Name': 'Who is the invoice billed to?',
    'Invoice Date': 'What is the invoice date?',
    'Due Date': 'When is the payment due?',
    'Total Amount': 'What is the total amount?',
    'Tax Amount': 'What is the tax amount?',
    'Payment Terms': 'What are the payment terms?',
}

print('=== Structured Invoice Extraction ===')
extract_document_fields(invoice_image, invoice_fields, doc_qa_pipeline)

## Performance Benchmarking

In [ ]:
def benchmark_doc_qa(
    image: Image.Image,
    questions: list[str],
    doc_pipe: transformers.Pipeline,
    num_runs: int = 3,
) -> pd.DataFrame:
    """Benchmark document QA speed.

    Args:
        image: Document image.
        questions: Questions to benchmark.
        doc_pipe: Document QA pipeline.
        num_runs: Number of runs.

    Returns:
        DataFrame with timing results.
    """
    # Warmup
    doc_pipe(image=image, question=questions[0])
    
    times: list[float] = []
    for _ in range(num_runs):
        start = time.perf_counter()
        for q in questions:
            doc_pipe(image=image, question=q)
        times.append(time.perf_counter() - start)
    
    return pd.DataFrame([{
        'Model': MODEL_NAME.split('/')[-1],
        'Questions': len(questions),
        'Total Time (ms)': f'{np.mean(times) * 1000:.1f}',
        'Per Question (ms)': f'{np.mean(times) * 1000 / len(questions):.1f}',
        'Device': str(device),
    }])


print('=== Document QA Speed ===')
benchmark_doc_qa(
    invoice_image,
    ['What is the total?', 'Who is it billed to?', 'What is the date?'],
    doc_qa_pipeline,
)

## Exercises

1. **Custom documents**: Create your own document images (business card, form, letter) and test the QA model on them.

2. **Donut comparison**: Load `naver-clova-ix/donut-base-finetuned-docvqa` and compare its answers with LayoutLM. Donut doesn't need OCR — does it perform better or worse?

3. **Real documents**: Scan or photograph a real receipt or invoice. How well does the model handle real-world document quality (wrinkles, shadows, skew)?

4. **Batch processing**: Build a pipeline that processes a folder of document images and extracts a standardized set of fields from each.

## Key Takeaways

- **Document Understanding** combines OCR, layout analysis, and language understanding to extract meaning from documents
- **LayoutLM** uses text + spatial position + image features for layout-aware understanding
- **Donut** is an end-to-end model that reads documents directly without separate OCR
- **Document QA** is the most flexible approach — define questions as your extraction schema
- Layout and spatial position matter as much as text content for accurate extraction

## Next Steps & Resources

**Next section**: [05_01 — Ollama Integration](../05_best_practices/05_01_ollama_integration.ipynb) — move to best practices and production topics.

**Resources:**
- [LayoutLM Paper](https://arxiv.org/abs/1912.13318) — Pre-training of Text and Layout
- [LayoutLMv3 Paper](https://arxiv.org/abs/2204.08387) — Unified Multimodal Pre-training
- [Donut Paper](https://arxiv.org/abs/2111.15664) — OCR-free Document Understanding Transformer
- [Document QA Models on Hub](https://huggingface.co/models?pipeline_tag=document-question-answering)
- [HuggingFace Document AI](https://huggingface.co/docs/transformers/tasks/document_question_answering)